
Notebook Name : 04-Transform-DataLake-Geocoding

GeoLocation JSON Source File Path : "abfss://bronze@datalakestorageaccountname.dfs.core.windows.net/geo-location/
"

In [0]:
geo_location_source_layer_name = 'adbbronze'
geo_location_source_storage_account_name = 'adbstorageu'
geo_location_source_folder_name = 'geo-location'

geo_location_source_folder_path = f"abfss://{geo_location_source_layer_name}@{geo_location_source_storage_account_name}.dfs.core.windows.net/{geo_location_source_folder_name}"

In [0]:
geo_location_bronze_DF = (spark
                       .read
                       .json(geo_location_source_folder_path))

In [0]:
display(geo_location_bronze_DF)

In [0]:
from pyspark.sql.functions import *

geo_location_silver_DF = ( geo_location_bronze_DF
                       .select(col('results.admin1').alias('state_name')
                              ,col('results.admin2').alias('district_name')
                               , col('results.country').alias('country_name')
                               , col('results.latitude').alias('latitude')
                               , col('results.longitude').alias('longitude')
                               , col('results.name').alias('market_name')
                               , col('results.population').alias('population')
)
)

display(geo_location_silver_DF)

In [0]:
geo_location_state_transDF = (geo_location_silver_DF
 .select(explode("state_name")
 ,monotonically_increasing_id().alias('state_sequence_id')))

 ##When we have array as a data type for each row value for each column, dont use explode on all the columns in the same data frame. Need to create one data frame using explode for each column

In [0]:

geo_location_state_trans_DF = ( geo_location_silver_DF
 .select(explode("state_name").alias('state_name')
 ,monotonically_increasing_id().alias('state_sequence_id')))

geo_location_district_trans_DF = (geo_location_silver_DF
 .select(explode("district_name").alias('district_name')
         ,monotonically_increasing_id().alias('district_sequence_id')
))
geo_location_country_trans_DF = (geo_location_silver_DF
 .select(explode("country_name").alias('country_name')
 ,monotonically_increasing_id().alias('country_name_sequence_id')))

geo_location_latitude_trans_DF = (geo_location_silver_DF
 .select(explode("latitude").alias('latitude')
 ,monotonically_increasing_id().alias('latitude_sequence_id')))

geo_location_longitude_trans_DF = (geo_location_silver_DF
 .select(explode("longitude").alias('longitude')
 ,monotonically_increasing_id().alias('longitude_sequence_id')))

geo_location_market_trans_DF = (geo_location_silver_DF
 .select(explode("market_name").alias('market_name')
 ,monotonically_increasing_id().alias('market_sequence_id')))

geo_location_population_trans_DF = (geo_location_silver_DF
 .select(explode("population").alias('population')
 ,monotonically_increasing_id().alias('population_sequence_id')))

In [0]:
geo_location_silver_trans_df = ( geo_location_state_trans_DF
                            .join(geo_location_district_trans_DF, col("state_sequence_id") == col("district_sequence_id"))
                             .join(geo_location_country_trans_DF, col("state_sequence_id") == col("country_name_sequence_id")) 
                            .join(geo_location_latitude_trans_DF, col("state_sequence_id") == col("latitude_sequence_id")) 
                            .join(geo_location_longitude_trans_DF, col("state_sequence_id") == col("longitude_sequence_id")) 
                            .join(geo_location_market_trans_DF, col("state_sequence_id") == col("market_sequence_id")) 
                            .join(geo_location_population_trans_DF, col("state_sequence_id") == col("population_sequence_id"))  
                                 .select(col("state_name")
                                    ,col("district_name")
                                  ,col("country_name")
                                    ,col("latitude")
                                    ,col("longitude")
                                    ,col("market_name")
                                    ,col("population")

                             ) )
#display(geoLocationSilverTransDF)                             

In [0]:
(geo_location_silver_trans_df 
.write
.mode('overwrite')
.saveAsTable("adb_rtp.silver.geo_location_silver"))

In [0]:
%sql
select * from adb_rtp.silver.geo_location_silver geolocation where geolocation.market_name = 'Guntur'

In [0]:
%sql
SELECT * FROM adb_rtp.silver.daily_pricing_silver dailyPricing WHERE dailyPricing.MARkET_NAME = 'Guntur'

SELECT DISTINCT geolocation.*
 FROM pricing_analytics.silver.geo_location_silver geolocation
inner join pricing_analytics.silver.daily_pricing_silver dailyPricing
on geolocation.stateName = dailyPricing.STATE_NAME
and geolocation.marketName = dailyPricing.MARKET_NAME
where geolocation.countryName = 'India'